# Nakshatra — Free AI Jewellery Try-On (Colab)

This notebook runs an experimental, generative SDXL inpainting workflow on a free Colab GPU when one is available. It uses the customer photo as the base image and the catalogue photo as an IP-Adapter reference. **This is real generative editing, not a pasted PNG overlay.**

1. Open this notebook in Google Colab and choose Runtime → Change runtime type → T4 GPU (if available).
2. Run the install cell, then the model cell. The first model download is large and can take time.
3. Run the app cell. Open the Gradio link. Upload the customer photo in the editor, paint a small mask over the exact finger/neck/ear/wrist area, upload the jewellery product photo, choose its type, and generate.

Free Colab sessions are temporary and not guaranteed. This notebook does not deploy a permanent public API. Review results carefully: generative models can alter small jewellery details and are not guaranteed to reproduce a product exactly.

In [ ]:
!pip -q install diffusers==0.35.1 transformers==4.56.2 accelerate==1.10.1 safetensors==0.6.2 huggingface_hub==0.36.0 gradio==5.49.1

In [ ]:
import torch, gradio as gr, numpy as np, gc
from PIL import Image, ImageFilter
from diffusers import AutoPipelineForInpainting

assert torch.cuda.is_available(), 'GPU not detected. In Colab select Runtime → Change runtime type → T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

MODEL_ID = 'diffusers/stable-diffusion-xl-1.0-inpainting-0.1'
pipe = AutoPipelineForInpainting.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe.load_ip_adapter(
    'h94/IP-Adapter',
    subfolder='sdxl_models',
    weight_name='ip-adapter-plus_sdxl_vit-h.safetensors',
    image_encoder_folder='models/image_encoder',
)
pipe.set_ip_adapter_scale(0.75)
pipe.enable_model_cpu_offload()
pipe.set_progress_bar_config(disable=True)
print('MODEL READY — SDXL Inpainting + IP-Adapter')

In [ ]:
def fit_image(im, max_side=768):
    im = im.convert('RGB')
    scale = min(1.0, max_side / max(im.size))
    w = max(64, int(im.width * scale) // 8 * 8)
    h = max(64, int(im.height * scale) // 8 * 8)
    return im.resize((w, h), Image.Resampling.LANCZOS)

def layer_to_mask(editor, size):
    if not isinstance(editor, dict) or editor.get('background') is None:
        raise gr.Error('Upload your customer photo in the editor first.')
    layers = editor.get('layers') or []
    if not layers:
        raise gr.Error('Paint the target jewellery area with the brush. The painted area is the AI edit mask.')
    mask = Image.new('L', size, 0)
    for layer in layers:
        if layer is None: continue
        lay = layer.convert('RGBA').resize(size, Image.Resampling.NEAREST)
        alpha = lay.getchannel('A')
        # Use visible brush strokes as the inpainting mask.
        mask = Image.fromarray(np.maximum(np.asarray(mask), np.asarray(alpha)))
    mask = mask.point(lambda p: 255 if p > 12 else 0).filter(ImageFilter.MaxFilter( nine := 9 )).filter(ImageFilter.GaussianBlur(2))
    if np.asarray(mask).max() < 20 or np.asarray(mask).mean() < 0.05:
        raise gr.Error('No painted mask found. Brush over the finger/neck/ear/wrist area where the jewellery should appear.')
    return mask

PROMPTS = {
 'Ring': 'Place the ring shown in the product reference image naturally on the selected finger area. Follow the finger direction, anatomy, perspective and scale. Preserve the exact visible gemstone colour, setting, halo, band shape and metal colour as closely as possible.',
 'Necklace': 'Place the necklace shown in the product reference naturally around the neck and collarbone inside the selected area. Follow body curvature, gravity, perspective and scale. Preserve its visible design, pendant, links and metal colour as closely as possible.',
 'Earrings': 'Place the earrings shown in the product reference naturally on the visible earlobe area selected by the mask. Follow ear anatomy, gravity, scale and perspective. Preserve the visible design and stones as closely as possible.',
 'Bracelet': 'Place the bracelet shown in the product reference naturally around the selected wrist area. Follow wrist anatomy, perspective and scale. Preserve the visible links, stones, clasp and metal colour as closely as possible.'
}

def generate(editor, product, category, steps, adapter_strength):
    if product is None: raise gr.Error('Upload a clear catalogue jewellery image.')
    if not torch.cuda.is_available(): raise gr.Error('Colab GPU is not available. Reconnect with a GPU runtime.')
    customer = fit_image(editor['background'])
    mask = layer_to_mask(editor, customer.size)
    product = product.convert('RGB')
    product.thumbnail((768, 768), Image.Resampling.LANCZOS)
    pipe.set_ip_adapter_scale(float(adapter_strength))
    prompt = (
      'Photorealistic jewellery virtual try-on image edit. The customer photo is the base image. '
      + PROMPTS[category] + ' '
      + 'Only modify pixels inside the white mask. Keep the same person, skin tone, face, hand, fingers, body, clothes, pose, background and lighting unchanged. '
      + 'Blend the jewellery into the skin with realistic contact, occlusion, reflections, soft shadows and natural highlights. '
      + 'Do not create extra fingers, extra jewellery, text, watermark, collage or border. Do not redesign the jewellery.'
    )
    negative = 'sticker, pasted-on look, floating jewellery, wrong finger, extra fingers, deformed hand, duplicate ring, extra jewellery, cartoon, illustration, text, watermark, blurry, low quality'
    with torch.inference_mode():
        result = pipe(
          prompt=prompt,
          negative_prompt=negative,
          image=customer,
          mask_image=mask,
          ip_adapter_image=product,
          strength=0.99,
          num_inference_steps=int(steps),
          guidance_scale=6.0,
          width=customer.width,
          height=customer.height,
        ).images[0]
    return result

with gr.Blocks(theme=gr.themes.Soft(), title='Nakshatra AI Jewellery Try-On') as demo:
    gr.Markdown('# Nakshatra — AI Jewellery Try-On\nUpload your photo, paint the exact target area, then upload the jewellery reference. The model generates a new edited image.')
    with gr.Row():
        with gr.Column():
            editor = gr.ImageEditor(label='1. Customer photo — upload, then paint over the target area', type='pil', brush=gr.Brush(colors=['#FFFFFF'], color_mode='fixed', default_size=24), eraser=gr.Eraser(), height=520)
            product = gr.Image(label='2. Original catalogue jewellery photo', type='pil')
            category = gr.Dropdown(['Ring','Necklace','Earrings','Bracelet'], value='Ring', label='Jewellery type')
            steps = gr.Slider(20, 40, value=30, step=5, label='Quality / speed (more steps = slower)')
            adapter = gr.Slider(0.45, 1.0, value=0.75, step=0.05, label='Product reference strength')
            run = gr.Button('Generate real AI try-on', variant='primary')
        with gr.Column():
            output = gr.Image(label='AI-generated result', type='pil', height=520)
            status = gr.Markdown('**Tip:** For a ring, paint a small band-shaped area across the ring finger where the ring should sit. Do not paint the whole hand.')
    run.click(generate, inputs=[editor, product, category, steps, adapter], outputs=output)

demo.queue(max_size=2).launch(share=True, debug=True)